<a href="https://colab.research.google.com/github/Devil-92/100_Days_of_ML/blob/main/USAdultCensusData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import shutil
import kagglehub

arr_path = kagglehub.dataset_download("uciml/adult-census-income")

print("Path to dataset files:", arr_path)

dtn_path = "/content/drive/MyDrive/Projects/USAdultCensus/Input"


for file_name in os.listdir(arr_path):

    src_file = os.path.join(arr_path, file_name)
    dst_file = os.path.join(dtn_path, file_name)

    if os.path.isfile(src_file):
        shutil.copy2(src_file, dst_file)

print("Files in input folder:", os.listdir(dtn_path))

Using Colab cache for faster access to the 'adult-census-income' dataset.
Path to dataset files: /kaggle/input/adult-census-income
Files in input folder: ['.ipynb_checkpoints', 'adult.csv', 'train_folds.csv']


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import pandas as pd

In [4]:
df = pd.read_csv('/content/drive/MyDrive/Projects/USAdultCensus/Input/adult.csv')

In [5]:
df.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


In [6]:
df.columns

Index(['age', 'workclass', 'fnlwgt', 'education', 'education.num',
       'marital.status', 'occupation', 'relationship', 'race', 'sex',
       'capital.gain', 'capital.loss', 'hours.per.week', 'native.country',
       'income'],
      dtype='object')

In [7]:
df.income.value_counts()

,count
income,
<=50K,24720
>50K,7841


In [8]:
%cd "/content/drive/MyDrive/Projects/USAdultCensus/src"

/content/drive/MyDrive/Projects/USAdultCensus/src


In [9]:
open('StratifiedKFold.py' , 'w').close()

In [10]:
%%writefile StratifiedKFold.py

import pandas as pd
from sklearn import model_selection

if __name__ == "__main__" :
  df = pd.read_csv('/content/drive/MyDrive/Projects/USAdultCensus/Input/adult.csv')
  df['kfold'] = -1
  df = df.sample(frac = 1).reset_index(drop = True)

  y = df.income.values

  kf = model_selection.KFold(n_splits = 5)

  for f , (tra_ , val_) in enumerate(kf.split(X = df , y = y)) :
    df.loc[val_ , 'kfold'] = f

  df.to_csv('/content/drive/MyDrive/Projects/USAdultCensus/Input/train_folds.csv' , index = False)

Overwriting StratifiedKFold.py


In [11]:
! python StratifiedKFold.py

### One Hot Encoding On Log Reg

In [12]:
open("ohe_logres.py" , 'a').close()

In [13]:
%%writefile ohe_logres.py
import pandas as pd

from sklearn import linear_model
from sklearn import preprocessing
from sklearn import metrics


def run(fold):

    df = pd.read_csv('/content/drive/MyDrive/Projects/USAdultCensus/Input/train_folds.csv')

    num_cols = [
        c for c in df.select_dtypes(include=['number']).columns
        if c != 'kfold'
    ]

    df = df.drop(num_cols, axis=1)

    target_mapping = {
        '<=50K': 0,
        '>50K': 1
    }

    df['income'] = df.income.map(target_mapping)

    features = [
        f for f in df.columns if f not in ('kfold', 'income')
    ]

    for f in features:
        df[f] = df[f].fillna('NONE').astype(str)

    df_train = df[df.kfold != fold].reset_index(drop=True)

    df_valid = df[df.kfold == fold].reset_index(drop=True)

    ohe = preprocessing.OneHotEncoder()

    full_data = pd.concat(
        [df_train, df_valid],
        axis=0
    )

    ohe.fit(full_data[features])

    X_train = ohe.transform(df_train[features])

    X_valid = ohe.transform(df_valid[features])

    model = linear_model.LogisticRegression(max_iter=1000)

    model.fit(X_train, df_train.income.values)

    valid_preds = model.predict_proba(X_valid)[:, 1]

    auc = metrics.roc_auc_score(df_valid.income.values, valid_preds)

    print(f'AUC for fold number {fold} is {auc}')


if __name__ == '__main__':
    for i in range(5):
        run(i)

Overwriting ohe_logres.py


In [14]:
num_cols = df.select_dtypes(include=['number']).columns.tolist()

In [15]:
num_cols

['age',
 'fnlwgt',
 'education.num',
 'capital.gain',
 'capital.loss',
 'hours.per.week']

In [16]:
df.income.value_counts()

,count
income,
<=50K,24720
>50K,7841


In [17]:
df.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


In [18]:
! python -W ignore ohe_logres.py

AUC for fold number 0 is 0.8794415449537908
AUC for fold number 1 is 0.8772611750829065
AUC for fold number 2 is 0.8837962628746558
AUC for fold number 3 is 0.8740949470345125
AUC for fold number 4 is 0.8806161667381779


## Label Encoding with XGBoost ( Only Cat Columns )

In [19]:
open('lbl_xgb.py' , 'a').close()

In [20]:
%%writefile lbl_xgb.py
import pandas as pd

from sklearn import preprocessing
from sklearn import metrics

import xgboost as xgb


def run(fold):
    df = pd.read_csv('/content/drive/MyDrive/Projects/USAdultCensus/Input/train_folds.csv')

    num_cols = ['age', 'fnlwgt', 'education.num', 'capital.gain', 'capital.loss', 'hours.per.week']

    df = df.drop(num_cols, axis=1)

    target_mapping = {
        '<=50K': 0,
        '>50K': 1
    }

    df['income'] = df.income.map(target_mapping)

    features = [
        f for f in df.columns if f not in ('kfold', 'income')
    ]

    for col in features:
        df[col] = df[col].fillna('NONE').astype(str)

    for col in features:
        lbl = preprocessing.LabelEncoder()
        lbl.fit(df[col])
        df[col] = lbl.transform(df[col])

    df_train = df[df.kfold != fold].reset_index(drop=True)

    df_valid = df[df.kfold == fold].reset_index(drop=True)

    x_train = df_train[features]

    x_valid = df_valid[features]

    model = xgb.XGBClassifier(
        n_jobs=-1,
        n_estimators = 200
    )

    model.fit(x_train, df_train.income.values)

    valid_pred = model.predict_proba(x_valid)[:, 1]

    auc = metrics.roc_auc_score(df_valid.income.values, valid_pred)

    print(f'AUC for fold number {fold} is {auc}')


if __name__ == '__main__':
    for i in range(5):
        run(i)

Overwriting lbl_xgb.py


In [21]:
! python -W ignore lbl_xgb.py

AUC for fold number 0 is 0.8711140179052677
AUC for fold number 1 is 0.8691443954470992
AUC for fold number 2 is 0.8739779377920589
AUC for fold number 3 is 0.8669842578653937
AUC for fold number 4 is 0.8722497432096838


In [22]:
print('Lets make the max number of trees to 200')

Lets make the max number of trees to 200


In [23]:
! python -W ignore lbl_xgb.py

AUC for fold number 0 is 0.8711140179052677
AUC for fold number 1 is 0.8691443954470992
AUC for fold number 2 is 0.8739779377920589
AUC for fold number 3 is 0.8669842578653937
AUC for fold number 4 is 0.8722497432096838


In [24]:
print("It looks like it doesn't improve")

It looks like it doesn't improve


## Label Encoding with XGBoost ( Both Cat Columns + Num Columns )

In [25]:
%%writefile lbl_xgb.py
import pandas as pd

from sklearn import preprocessing
from sklearn import metrics

import xgboost as xgb


def run(fold):
    df = pd.read_csv('/content/drive/MyDrive/Projects/USAdultCensus/Input/train_folds.csv')

    num_cols = ['age', 'fnlwgt', 'education.num', 'capital.gain', 'capital.loss', 'hours.per.week']

    # df = df.drop(num_cols, axis=1)

    target_mapping = {
        '<=50K': 0,
        '>50K': 1
    }

    df['income'] = df.income.map(target_mapping)

    features = [
        f for f in df.columns if f not in ('kfold', 'income')
    ]

    for col in features:
        df[col] = df[col].fillna('NONE').astype(str)

    for col in features:
        lbl = preprocessing.LabelEncoder()
        lbl.fit(df[col])
        df[col] = lbl.transform(df[col])

    df_train = df[df.kfold != fold].reset_index(drop=True)

    df_valid = df[df.kfold == fold].reset_index(drop=True)

    x_train = df_train[features]

    x_valid = df_valid[features]

    model = xgb.XGBClassifier(
        n_jobs=-1,
        n_estimators = 200
    )

    model.fit(x_train, df_train.income.values)

    valid_pred = model.predict_proba(x_valid)[:, 1]

    auc = metrics.roc_auc_score(df_valid.income.values, valid_pred)

    print(f'AUC for fold number {fold} is {auc}')


if __name__ == '__main__':
    for i in range(5):
        run(i)

Overwriting lbl_xgb.py


In [26]:
print('Now lets include the numerical Features in xgboost model')

Now lets include the numerical Features in xgboost model


In [27]:
! python -W ignore lbl_xgb.py

AUC for fold number 0 is 0.9194554800373355
AUC for fold number 1 is 0.9193976107791166
AUC for fold number 2 is 0.922367239431096
AUC for fold number 3 is 0.9142458375254363
AUC for fold number 4 is 0.9242306042048601


In [28]:
print('We keep out numerical column as it is and encode the categorical column , any tree based model can handle this mix')

We keep out numerical column as it is and encode the categorical column , any tree based model can handle this mix


## Label Encoding With XGBoost ( With Combinations for Cat Columns )

In [29]:
open('lbl_xgb_num_feat.py' , 'a').close()

In [30]:
%%writefile lbl_xgb_num_feat.py
import itertools
import xgboost as xgb
import pandas as pd

from sklearn import preprocessing
from sklearn import metrics


def feature_engineering(df, cat_cols):

    combi = list(itertools.combinations(cat_cols, 2))

    for c1, c2 in combi:
        df.loc[
            :,
            c1 + '_' + c2
        ] = df[c1].astype(str) + '_' + df[c2].astype(str)

    return df


def run(fold):
    df = pd.read_csv('/content/drive/MyDrive/Projects/USAdultCensus/Input/train_folds.csv')

    num_cols = [c for c in df.select_dtypes('number').columns if c not in ('kfold',)]

    target_mapping = {
        '<=50K': 1,
        '>50K': 0
    }

    df['income'] = df.income.map(target_mapping)

    cat_cols = [c for c in df.columns if c not in num_cols and c not in ('kfold', 'income')]

    df = feature_engineering(df, cat_cols)

    feature = [
        f for f in df.columns if f not in ('kfold', 'income')
    ]

    for cols in feature:
        if cols not in num_cols:
            df[cols] = df[cols].fillna('NONE').astype(str)

    for cols in feature:
        if cols not in num_cols:
            lbl = preprocessing.LabelEncoder()

            lbl.fit(df[cols])

            df[cols] = lbl.transform(df[cols])

    df_train = df[df.kfold != fold].reset_index(drop=True)

    df_valid = df[df.kfold == fold].reset_index(drop=True)

    x_train = df_train[feature]

    x_valid = df_valid[feature]

    model = xgb.XGBClassifier(
        n_jobs=-1,
    )

    model.fit(x_train, df_train.income.values)

    valid_preds = model.predict_proba(x_valid)[:, 1]

    auc = metrics.roc_auc_score(df_valid.income.values, valid_preds)

    print(f'auc score for fold number {fold} is {auc}')


if __name__ == '__main__':
    for i in range(5):
        run(i)

Overwriting lbl_xgb_num_feat.py


In [31]:
! python lbl_xgb_num_feat.py

auc score for fold number 0 is 0.926791578476869
auc score for fold number 1 is 0.9251802721106063
auc score for fold number 2 is 0.9268203459114183
auc score for fold number 3 is 0.9171470249307201
auc score for fold number 4 is 0.9277691097502307


In [32]:
print('Lets change the max_depth to 8')

Lets change the max_depth to 8


In [33]:
%%writefile lbl_xgb_num_feat.py
import itertools
import xgboost as xgb
import pandas as pd

from sklearn import preprocessing
from sklearn import metrics


def feature_engineering(df, cat_cols):

    combi = list(itertools.combinations(cat_cols, 2))

    for c1, c2 in combi:
        df.loc[
            :,
            c1 + '_' + c2
        ] = df[c1].astype(str) + '_' + df[c2].astype(str)

    return df


def run(fold):
    df = pd.read_csv('/content/drive/MyDrive/Projects/USAdultCensus/Input/train_folds.csv')

    num_cols = [c for c in df.select_dtypes('number').columns if c not in ('kfold',)]

    target_mapping = {
        '<=50K': 1,
        '>50K': 0
    }

    df['income'] = df.income.map(target_mapping)

    cat_cols = [c for c in df.columns if c not in num_cols and c not in ('kfold', 'income')]

    df = feature_engineering(df, cat_cols)

    feature = [
        f for f in df.columns if f not in ('kfold', 'income')
    ]

    for cols in feature:
        if cols not in num_cols:
            df[cols] = df[cols].fillna('NONE').astype(str)

    for cols in feature:
        if cols not in num_cols:
            lbl = preprocessing.LabelEncoder()

            lbl.fit(df[cols])

            df[cols] = lbl.transform(df[cols])

    df_train = df[df.kfold != fold].reset_index(drop=True)

    df_valid = df[df.kfold == fold].reset_index(drop=True)

    x_train = df_train[feature]

    x_valid = df_valid[feature]

    model = xgb.XGBClassifier(
        n_jobs=-1,
        max_depth = 8 ,
    )

    model.fit(x_train, df_train.income.values)

    valid_preds = model.predict_proba(x_valid)[:, 1]

    auc = metrics.roc_auc_score(df_valid.income.values, valid_preds)

    print(f'auc score for fold number {fold} is {auc}')


if __name__ == '__main__':
    for i in range(5):
        run(i)

Overwriting lbl_xgb_num_feat.py


In [34]:
! python lbl_xgb_num_feat.py

auc score for fold number 0 is 0.9208153995819476
auc score for fold number 1 is 0.9217577061986968
auc score for fold number 2 is 0.9240635299020715
auc score for fold number 3 is 0.914711948943338
auc score for fold number 4 is 0.9244387668864011


## Target Encoding with XGBoost



> One more way of feature engineering from categorical features is to use target
encoding. However, you have to be very careful here as this might overfit your
model. Target encoding is a technique in which you map each category in a given
feature to its mean target value, but this must always be done in a cross-validated
manner. It means that the first thing you do is create the folds, and then use those
folds to create target encoding features for different columns of the data in the same
way you fit and predict the model on folds. So, if you have created 5 folds, you
have to create target encoding 5 times such that in the end, you have encoding for
variables in each fold which are not derived from the same fold. And then when
you fit your model, you must use the same folds again. Target encoding for unseen
test data can be derived from the full training data or can be an average of all the 5
folds



In [35]:
open('target_encoding.py' , 'a').close()

In [36]:
%%writefile target_encoding.py
import pandas as pd

from sklearn import metrics
from sklearn import preprocessing
import xgboost as xgb

def mean_target_encoding(data) :
  df = data.copy()

  num_cols = [f for f in df.select_dtypes('number') if f not in ('kfold')]

  target_mapping = {
      '<=50K' : 0 ,
      '>50K' : 1
  }

  df['income'] = df.income.map(target_mapping)

  features = [
      f for f in df.columns if f not in ('kfold' , 'income') and f not in num_cols
  ]

  for col in features :
    if col not in num_cols :
      df[col] = df[col].astype(str).fillna('NONE')

  for col in features :
    if col not in num_cols :
      lbl = preprocessing.LabelEncoder()
      lbl.fit(df[col])
      df[col] = lbl.transform(df[col])

  encoded_dfs = []

  for i in range(5) :
    df_train = df[df.kfold != i].reset_index(drop = True)

    df_valid = df[df.kfold == i].reset_index(drop = True)

    for col in features :
      mapping_dict = dict(
          df_train.groupby(col)['income'].mean()
      )

      df_valid.loc[: , col + '_enc'] = df_valid[col].map(mapping_dict)

    encoded_dfs.append(df_valid)

  encoded_df = pd.concat(encoded_dfs , axis = 0)

  return encoded_df

def run(df , fold) :

  df_train = df[df.kfold != fold].reset_index(drop = True)

  df_valid = df[df.kfold == fold].reset_index(drop = True)

  features = [
      f for f in df.columns if f not in ('income' , 'kfold')
  ]

  x_train = df_train[features]

  x_valid = df_valid[features]

  model = xgb.XGBClassifier(
      n_jobs = -1 ,
      max_depth = 7
  )

  model.fit(x_train , df_train.income.values)

  valid_preds = model.predict_proba(x_valid)[: , 1]

  auc = metrics.roc_auc_score(df_valid.income.values , valid_preds)

  print(f'AUC for fold number {fold} is {auc}')

if __name__ == '__main__' :

  df = pd.read_csv('/content/drive/MyDrive/Projects/USAdultCensus/Input/train_folds.csv')

  df = mean_target_encoding(df)

  for i in range(5) :
    run(df , i)

Overwriting target_encoding.py




> It must be noted that in the above snippet, I had not dropped categorical columns
when I did the target encoding .
I kept all the features and added target encoded
features on top of it. Also, I used mean. You can use mean, median, standard
deviation or any other function of targets.





In [37]:
! python target_encoding.py

AUC for fold number 0 is 0.9249431422298762
AUC for fold number 1 is 0.9252458303167057
AUC for fold number 2 is 0.9232602798850479
AUC for fold number 3 is 0.9072074542524946
AUC for fold number 4 is 0.9251923652013367




>However, you must be very careful
when using target encoding as it is too prone to overfitting. When we use target
encoding, it’s better to use some kind of smoothing or adding noise in the encoded
values. Scikit-learn has contrib repository which has target encoding with
smoothing, or you can create your own smoothing. Smoothing introduces some
kind of regularization that helps with not overfitting the model. It’s not very
difficult.



### Target Encoding

### Core Overview
* **What it is:** Replace a categorical value with a statistic (usually mean) of the target for that category.
* **Why:** Avoids high-dimensional one-hot blowup on high-cardinality features; encodes actual predictive signal instead of treating categories as unrelated.
* **Core risk:** Computing the mean using the same rows you train on leaks the label into the feature $\rightarrow$ overfitting, especially for rare categories.

---

### Fix : Out-of-Fold Encoding

1. **Split data into K folds**
2. **For each fold**, compute category means from the *other* folds only, apply to that fold.
3. **Repeat for all folds** $\rightarrow$ every row's encoding is derived from data it wasn't part of.
4. **Train your model** using the same fold structure.
5. **Test/unseen data:** Encode using either:
   * **(a)** Category means from the full training set, or
   * **(b)** Average of the $K$ fold-wise mappings.

## Entity Encoding with Neural Network



> In entity embeddings, the
categories are represented as vectors. We represent categories by vectors in both
binarization and one hot encoding approaches. But what if we have tens of
thousands of categories. This will create huge matrices and will take a long time for
us to train complicated models. We can thus represent them by vectors with float
values instead.





> The idea is super simple. You have an embedding layer for each categorical feature.
So, every category in a column can now be mapped to an embedding (like mapping
words to embeddings in natural language processing). You then reshape these
embeddings to their dimension to make them flat and then concatenate all the
flattened inputs embeddings. Then add a bunch of dense layers, an output layer and
you are done.



In [38]:
print('When embedding size is the same as the number of unique categories, we have one-hot-encoding.')

When embedding size is the same as the number of unique categories, we have one-hot-encoding.


In [39]:
open('entity_embeddings.py' , 'a').close()

In [40]:
! pwd

/content/drive/MyDrive/Projects/USAdultCensus/src


In [41]:
import tensorflow

In [42]:
%%writefile entity_embeddings.py
import os
import joblib
import pandas as pd
import numpy as np

from sklearn import metrics , preprocessing
from tensorflow.keras import layers
from tensorflow.keras import optimizers
from tensorflow.keras import Model
from tensorflow.keras import callbacks
from tensorflow.keras import backend as K
from tensorflow.keras import utils

def create_model(data , catcols) :

  inputs = []

  outputs = []

  for c in catcols :

    num_unique_values = int(len(np.unique(data[c])))

    emb_dim = int(min(np.ceil(num_unique_values/2) , 50))      ## Generalized shape of embed_dim

    inp = layers.Input(shape = (1 , ))        ## Actual [1 , None] , Here None is batch_size

    out = layers.Embedding(
        num_unique_values + 1 , emb_dim , name = c      ## [num_unique + 1 , embed_dim]
    )(inp)

    out = layers.SpatialDropout1D(0.3)(out)

    out = layers.Reshape(target_shape = (emb_dim , ))(out)      ## [None , embed_dim]

    inputs.append(inp)

    outputs.append(out)

  x = layers.Concatenate()(outputs)

  x = layers.BatchNormalization()(x)

  x = layers.Dense(300 , activation = 'relu')(x)        ## [None , 300]
  x = layers.Dropout(0.3)(x)
  x = layers.BatchNormalization()(x)

  x = layers.Dense(300 , activation = 'relu')(x)
  x = layers.Dropout(0.3)(x)
  x = layers.BatchNormalization()(x)

  y = layers.Dense(2 , activation = 'softmax')(x)     ## [None , 2]

  model = Model(inputs = inputs , outputs = y)

  model.compile(loss = 'binary_crossentropy' , optimizer = 'adam')

  return model

def run(fold) :

  df = pd.read_csv('/content/drive/MyDrive/Projects/USAdultCensus/Input/train_folds.csv')

  features = [
      f for f in df.columns if f not in ('kfold' , 'id' , 'income')
  ]

  target_mapping = {
      '<=50K': 0 ,
      '>50K': 1
  }

  df['income'] = df.income.map(target_mapping)

  for c in features :
    df[c] = df[c].astype(str).fillna('NONE')

  for f in features :
    lbl = preprocessing.LabelEncoder()

    df[f] = lbl.fit_transform(df[f])

  df_train = df[df.kfold != fold].reset_index(drop = True)

  df_valid = df[df.kfold == fold].reset_index(drop = True)

  model = create_model(df , features)

  xtrain = [
      df_train[features].values[: , k] for k in range(len(features))
  ]

  xvalid = [
      df_valid[features].values[: , k] for k in range(len(features))
  ]

  ytrain = df_train.income.values

  yvalid = df_valid.income.values

  ytrain_cat = utils.to_categorical(ytrain)

  yvalid_cat = utils.to_categorical(yvalid)

  model.fit(
      xtrain ,
      ytrain_cat ,  ## As our model will output [x , y] shape vector
      validation_data = (xvalid , yvalid_cat) ,   ## validation data only takes tupples
      verbose = 1 ,
      batch_size = 1024 ,
      epochs = 3
  )

  valid_preds = model.predict(xvalid)[: , 1]

  auc = metrics.roc_auc_score(yvalid , valid_preds)

  print(f'AUC score for fold number {fold} is {auc}')

  K.clear_session()

if __name__ == '__main__' :
  for i in range(5) :
    run(i)

Overwriting entity_embeddings.py


In [43]:
! python entity_embeddings.py

2026-09-29 21:19:56.983594: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
Epoch 1/3
26/26 ━━━━━━━━━━━━━━━━━━━━ 14s 158ms/step - loss: 0.6184 - val_loss: 0.5775
Epoch 2/3
26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 80ms/step - loss: 0.3847 - val_loss: 0.5383
Epoch 3/3
26/26 ━━━━━━━━━━━━━━━━━━━━ 3s 80ms/step - loss: 0.3014 - val_loss: 0.5573
204/204 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step
AUC score for fold number 0 is 0.8777961533910894
Epoch 1/3
26/26 ━━━━━━━━━━━━━━━━━━━━ 13s 112ms/step - loss: 0.6207 - val_loss: 0.5752
Epoch 2/3
26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 78ms/step - loss: 0.3742 - val_loss: 0.5463
Epoch 3/3
26/26 ━━━━━━━━━━━━━━━━━━━━ 4s 128ms/step - loss: 0.3053 - val_loss: 0.5674
204/204 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step
AUC score for fold number 1 is 0.8715564450972049
Epoch 1/3
26/26 ━━━━━━━━━━━━━━━━━━━━ 12s 153ms/step - loss: 0.6087 - val_loss: 0.5775
Epoch 2/3
26/26 ━━━━━━━━━━━━━━━━━━━━ 3s 